# 面试问题：Chunked Prefill 调度器怎样保护 Decode TPOT 与长 Prompt 公平？

可直接复述的回答：Monolithic Prefill 会让一个长 prompt 独占整轮计算，已经在 decode 的请求因此出现 TPOT 抖动。Chunked Prefill 把 prompt 切成固定或自适应 token 块，每轮先为 decode 预留预算，再推进 prefill。调度器同时检查 KV block 准入、绝对位置和租户公平。Chunk 太大仍会阻塞 decode，太小则增加启动与调度开销。评估要同时报告 TTFT、TPOT、长 prompt 完成时间和吞吐。取消和超时必须释放已分配 KV。真实系统还需依据模型和 GPU profile 搜索 chunk size。

后续实验使用可读的小型业务数据验证关键判断。所有数值都标记为教学实验，不代表真实 GPU、线上流量或基础模型泛化结果。


## 1. 真实案例：在线客服推理请求与输入预览

五条请求保留到达 tick、prompt token、输出 token 和租户。`long-doc` 在已有两个聊天请求 decode 时到达，用于观察长 prefill 对 TPOT 的干扰。


In [1]:
requests06 = [  # 构造五条带到达时间和长度的推理请求。
    {"id": "chat-a", "tenant": "shop-a", "arrival": 0, "prompt": 16, "output": 5},  # 早到短聊天。
    {"id": "chat-b", "tenant": "shop-b", "arrival": 0, "prompt": 20, "output": 5},  # 第二个早到聊天。
    {"id": "long-doc", "tenant": "shop-c", "arrival": 2, "prompt": 120, "output": 3},  # 中途到达的长文档。
    {"id": "chat-c", "tenant": "shop-a", "arrival": 4, "prompt": 12, "output": 4},  # 长文档期间到达的短请求。
    {"id": "chat-d", "tenant": "shop-d", "arrival": 5, "prompt": 18, "output": 4},  # 另一个短请求。
]  # 完成真实语义调度工作负载。
print("教学实验输入：id | tenant | arrival | prompt | output")  # 输出请求预览表头。
for request06 in requests06:  # 逐条展示调度字段。
    print(request06)  # 输出一条推理请求。


教学实验输入：id | tenant | arrival | prompt | output
{'id': 'chat-a', 'tenant': 'shop-a', 'arrival': 0, 'prompt': 16, 'output': 5}
{'id': 'chat-b', 'tenant': 'shop-b', 'arrival': 0, 'prompt': 20, 'output': 5}
{'id': 'long-doc', 'tenant': 'shop-c', 'arrival': 2, 'prompt': 120, 'output': 3}
{'id': 'chat-c', 'tenant': 'shop-a', 'arrival': 4, 'prompt': 12, 'output': 4}
{'id': 'chat-d', 'tenant': 'shop-d', 'arrival': 5, 'prompt': 18, 'output': 4}


## 2. Baseline（基线）：Monolithic Prefill

每 tick 预算 32 token。基线一旦选择 prefill 请求，就连续占用 `ceil(prompt/32)` 个 tick；decode 在这些 tick 完全暂停，直接造成 token 间隔尖峰。


In [2]:
def simulate06(chunk06, startup_cost06=0):  # 模拟带可配置 chunk 的 prefill/decode 调度。
    states06 = {request06["id"]: {"request": request06, "prefill_left": request06["prompt"], "generated": 0, "first": None, "decode_ticks": [], "done": None} for request06 in requests06}  # 初始化每个请求状态。
    events06 = []  # 收集逐 tick 调度事件。
    blocked_ticks06 = 0  # 记录跨多轮 prefill 尚需独占的 tick 数。
    blocked_request06 = None  # 记录当前独占 GPU 的 prefill 请求。
    for tick06 in range(80):  # 运行足够覆盖全部请求的离散时间。
        if blocked_ticks06 > 0:  # 检查上一轮大 chunk 是否仍在占用 GPU。
            events06.append((tick06, blocked_request06, "prefill_block", 0))  # 记录 decode 被阻塞的持续事件。
            blocked_ticks06 -= 1  # 消耗一个独占计算 tick。
            continue  # 本轮不允许任何 decode 或新 prefill。
        arrived06 = [state06 for state06 in states06.values() if state06["request"]["arrival"] <= tick06 and state06["done"] is None]  # 获取已经到达且未完成请求。
        decode06 = [state06 for state06 in arrived06 if state06["prefill_left"] == 0 and state06["generated"] < state06["request"]["output"]]  # 找出可执行 decode 的请求。
        for state06 in decode06:  # 先为所有活跃 decode 生成一个 token。
            state06["generated"] += 1  # 推进一个输出 token。
            state06["decode_ticks"].append(tick06)  # 记录 token 生成时间用于 TPOT。
            if state06["first"] is None:  # 检查是否为首个输出 token。
                state06["first"] = tick06  # 保存 TTFT 时间点。
            events06.append((tick06, state06["request"]["id"], "decode", 1))  # 保存 decode 事件。
            if state06["generated"] == state06["request"]["output"]:  # 检查请求是否完成生成。
                state06["done"] = tick06  # 保存完成 tick。
        waiting06 = [state06 for state06 in arrived06 if state06["prefill_left"] > 0]  # 找出等待 prefill 的请求。
        if waiting06:  # 检查本轮是否有 prompt 工作。
            waiting06.sort(key=lambda state06: (state06["request"]["arrival"], state06["request"]["id"]))  # 按到达顺序保证公平。
            state06 = waiting06[0]  # 选择最早到达的 prefill 请求。
            work06 = min(chunk06, state06["prefill_left"])  # 限制本轮最多处理一个 chunk。
            state06["prefill_left"] -= work06  # 推进 prompt token。
            events06.append((tick06, state06["request"]["id"], "prefill", work06 + startup_cost06))  # 记录 prefill 与启动开销。
            duration06 = (work06 + startup_cost06 + 31) // 32  # 按每tick 32 token计算当前chunk占用时长。
            blocked_ticks06 = max(0, duration06 - 1)  # 保存后续仍需独占的完整 tick 数。
            blocked_request06 = state06["request"]["id"]  # 保存阻塞事件的请求标识。
        if all(state06["done"] is not None for state06 in states06.values()):  # 检查全部请求是否完成。
            return states06, events06, tick06 + 1  # 返回状态、轨迹和总时间。
    return states06, events06, 80  # 返回超时前状态用于失败分析。
monolithic_states06, monolithic_events06, monolithic_time06 = simulate06(chunk06=120)  # 使用最大 prompt 大小模拟整段 prefill。
print("Monolithic前20个事件：tick | request | phase | work")  # 输出基线调度轨迹表头。
for event06 in monolithic_events06[:20]:  # 展示长 prompt 到达后的关键事件。
    print(event06)  # 输出一条调度事件。


Monolithic前20个事件：tick | request | phase | work
(0, 'chat-a', 'prefill', 16)
(1, 'chat-a', 'decode', 1)
(1, 'chat-b', 'prefill', 20)
(2, 'chat-a', 'decode', 1)
(2, 'chat-b', 'decode', 1)
(2, 'long-doc', 'prefill', 120)
(3, 'long-doc', 'prefill_block', 0)
(4, 'long-doc', 'prefill_block', 0)
(5, 'long-doc', 'prefill_block', 0)
(6, 'chat-a', 'decode', 1)
(6, 'chat-b', 'decode', 1)
(6, 'long-doc', 'decode', 1)
(6, 'chat-c', 'prefill', 12)
(7, 'chat-a', 'decode', 1)
(7, 'chat-b', 'decode', 1)
(7, 'long-doc', 'decode', 1)
(7, 'chat-c', 'decode', 1)
(7, 'chat-d', 'prefill', 18)
(8, 'chat-a', 'decode', 1)
(8, 'chat-b', 'decode', 1)


## 3. 核心实现：Decode 优先与 Chunked Prefill

教学模拟每 tick 先为所有活跃 decode 生成一个 token，再用剩余机会推进至多 24 个 prompt token。绝对位置等于原 prompt 长度减剩余量，不能在每个 chunk 重置。


In [3]:
chunk_size06 = 24  # 设置保护 decode 的 prefill chunk 大小。
chunked_states06, chunked_events06, chunked_time06 = simulate06(chunk06=chunk_size06)  # 运行 chunked prefill 调度。
position_trace06 = []  # 收集长文档每个 chunk 的绝对位置范围。
consumed06 = 0  # 初始化长文档已消费 prompt token。
for tick06, request_id06, phase06, work06 in chunked_events06:  # 遍历完整调度轨迹。
    if request_id06 == "long-doc" and phase06 == "prefill":  # 只观察长文档 prefill。
        start06 = consumed06  # 记录当前 chunk 的绝对起点。
        consumed06 += work06  # 推进全局 prompt 位置。
        position_trace06.append((tick06, start06, consumed06))  # 保存不重置的位置区间。
print("Chunked前24个事件：tick | request | phase | work")  # 输出核心调度轨迹表头。
for event06 in chunked_events06[:24]:  # 展示 decode 与 prefill 交错过程。
    print(event06)  # 输出一条 chunked 调度事件。
print("long-doc绝对位置轨迹", position_trace06)  # 展示 chunk 边界仍保持连续位置。


Chunked前24个事件：tick | request | phase | work
(0, 'chat-a', 'prefill', 16)
(1, 'chat-a', 'decode', 1)
(1, 'chat-b', 'prefill', 20)
(2, 'chat-a', 'decode', 1)
(2, 'chat-b', 'decode', 1)
(2, 'long-doc', 'prefill', 24)
(3, 'chat-a', 'decode', 1)
(3, 'chat-b', 'decode', 1)
(3, 'long-doc', 'prefill', 24)
(4, 'chat-a', 'decode', 1)
(4, 'chat-b', 'decode', 1)
(4, 'long-doc', 'prefill', 24)
(5, 'chat-a', 'decode', 1)
(5, 'chat-b', 'decode', 1)
(5, 'long-doc', 'prefill', 24)
(6, 'chat-b', 'decode', 1)
(6, 'long-doc', 'prefill', 24)
(7, 'long-doc', 'decode', 1)
(7, 'chat-c', 'prefill', 12)
(8, 'long-doc', 'decode', 1)
(8, 'chat-c', 'decode', 1)
(8, 'chat-d', 'prefill', 18)
(9, 'long-doc', 'decode', 1)
(9, 'chat-c', 'decode', 1)
long-doc绝对位置轨迹 [(2, 0, 24), (3, 24, 48), (4, 48, 72), (5, 72, 96), (6, 96, 120)]


## 4. 结果表、TPOT/TTFT 与结果解读

对每个请求计算首 token 延迟和最大 decode token 间隔。Chunked 方案应让已有聊天请求持续 decode，同时逐步推进长文档。


In [4]:
def metrics06(states06):  # 从请求状态计算 TTFT 与最大 TPOT。
    rows06 = []  # 收集逐请求延迟指标。
    for request_id06, state06 in states06.items():  # 遍历所有完成请求。
        arrival06 = state06["request"]["arrival"]  # 读取请求到达时间。
        ttft06 = state06["first"] - arrival06  # 计算首 token 延迟。
        gaps06 = [right06 - left06 for left06, right06 in zip(state06["decode_ticks"], state06["decode_ticks"][1:])]  # 计算相邻 decode token 间隔。
        max_tpot06 = max(gaps06) if gaps06 else 1  # 单 token 情况使用一个 tick。
        rows06.append((request_id06, ttft06, max_tpot06, state06["done"] - arrival06))  # 保存 TTFT、TPOT 和完成延迟。
    return rows06  # 返回逐请求指标表。
monolithic_metrics06 = metrics06(monolithic_states06)  # 计算整段 prefill 指标。
chunked_metrics06 = metrics06(chunked_states06)  # 计算 chunked 指标。
print("方法 | request | TTFT | max_TPOT | latency")  # 输出调度结果表头。
for row06 in monolithic_metrics06:  # 展示整段 prefill 结果。
    print("monolithic", row06)  # 输出一条基线延迟记录。
for row06 in chunked_metrics06:  # 展示 chunked prefill 结果。
    print("chunked", row06)  # 输出一条核心延迟记录。
print("结果解读：chunk让长prompt多轮推进，同时已有decode每tick都获得服务")  # 解释调度交错带来的收益。


方法 | request | TTFT | max_TPOT | latency
monolithic ('chat-a', 1, 4, 8)
monolithic ('chat-b', 2, 4, 9)
monolithic ('long-doc', 4, 1, 6)
monolithic ('chat-c', 3, 1, 6)
monolithic ('chat-d', 3, 1, 6)
chunked ('chat-a', 1, 1, 5)
chunked ('chat-b', 2, 1, 6)
chunked ('long-doc', 5, 1, 7)
chunked ('chat-c', 4, 1, 7)
chunked ('chat-d', 4, 1, 7)
结果解读：chunk让长prompt多轮推进，同时已有decode每tick都获得服务


## 5. 失败案例与修正：Chunk 越小启动开销越高

若每个 chunk 都有固定启动成本，过小 chunk 会增加总工作和长 prompt TTFT。下面搜索 8/16/24/48，比较最大 TPOT 与总调度工作，选择满足 TPOT 后开销最低者。


In [5]:
search_rows06 = []  # 收集不同 chunk size 的延迟与开销。
for candidate06 in [8, 16, 24, 48]:  # 遍历四个候选 chunk 大小。
    states06, events06, total_time06 = simulate06(chunk06=candidate06, startup_cost06=2)  # 加入每 chunk 两 token 等价启动开销。
    rows06 = metrics06(states06)  # 计算候选延迟指标。
    max_tpot06 = max(row06[2] for row06 in rows06)  # 汇总最差 decode token 间隔。
    scheduled_work06 = sum(event06[3] for event06 in events06)  # 汇总 token 工作和启动开销。
    search_rows06.append((candidate06, max_tpot06, scheduled_work06, total_time06))  # 保存候选结果。
feasible06 = [row06 for row06 in search_rows06 if row06[1] <= 1]  # 保留满足教学 TPOT 目标的候选。
selected_chunk06 = min(feasible06, key=lambda row06: (row06[2], row06[0]))[0]  # 在满足TPOT后选择调度工作最小者。
print("失败与修正：chunk | max_TPOT | scheduled_work | total_ticks")  # 输出 chunk 搜索表头。
for row06 in search_rows06:  # 展示过小 chunk 的启动开销。
    print(row06)  # 输出一个候选调度结果。
print("按TPOT门禁选择chunk", selected_chunk06)  # 展示约束优化而非盲目取最小值。


失败与修正：chunk | max_TPOT | scheduled_work | total_ticks
(8, 1, 257, 29)
(16, 1, 235, 18)
(24, 1, 225, 13)
(48, 2, 221, 13)
按TPOT门禁选择chunk 24


## 6. 生产边界与调度制品

真实调度器按 GPU 时间和 KV block 而非 token 简化值准入，还要处理取消、prefix cache、speculative token、租户配额和 prefill/decode 分离。


In [6]:
scheduler_contract06 = {"decode_priority": True, "chunk_candidates": [8, 16, 24, 48], "selected_chunk": selected_chunk06, "position": "absolute", "admission": "kv_blocks", "cancel_releases_kv": True}  # 定义调度发布合同。
print("Chunked Prefill 制品", scheduler_contract06)  # 展示 chunk、位置和 KV 语义。
print("生产替换点：实测GPU时间、Paged KV准入、取消、prefix cache、租户公平和在线chunk自适应")  # 说明离散 tick 模型的边界。


Chunked Prefill 制品 {'decode_priority': True, 'chunk_candidates': [8, 16, 24, 48], 'selected_chunk': 24, 'position': 'absolute', 'admission': 'kv_blocks', 'cancel_releases_kv': True}
生产替换点：实测GPU时间、Paged KV准入、取消、prefix cache、租户公平和在线chunk自适应


## 7. 最小回归测试

断言保护案例规模、位置连续性和 chunk 搜索。


In [7]:
assert len(requests06) >= 5  # 保证调度案例覆盖长短请求和多租户。
assert position_trace06[-1][2] == 120  # 保证长文档所有 prompt token 被完整推进。
assert all(right06[1] == left06[2] for left06, right06 in zip(position_trace06, position_trace06[1:]))  # 保证 chunk 间绝对位置连续。
assert all(row06[2] <= 1 for row06 in chunked_metrics06)  # 保证 chunked 调度保护 decode TPOT。
assert selected_chunk06 in {8, 16, 24, 48}  # 保证选择来自已评测候选。
print("最小回归测试通过：Decode优先、绝对位置和chunk搜索稳定")  # 显示调度关键性质已验证。


最小回归测试通过：Decode优先、绝对位置和chunk搜索稳定
